# ChakraModel - Final Kaggle Proof Evaluation

This notebook provides the definitive, undeniable proof of ChakraModel's performance on full-cohort OOD datasets and real-world video sequences from HyperKvasir. No truncation. No synthetic data.

## 1. Environment Setup

In [1]:
import os
import shutil

!rm -rf /kaggle/working/chakramodel
CODE_PATH = '/kaggle/input/chakramodel-kaggle-code'
if os.path.exists(CODE_PATH):
    shutil.copytree(CODE_PATH, '/kaggle/working/chakramodel', dirs_exist_ok=True)
    %cd /kaggle/working/chakramodel
    !pip install ultralytics thop gdown numpy opencv-python matplotlib > /dev/null
    print("Environment setup complete.")
else:
    print(f"CRITICAL ERROR: Code path {CODE_PATH} not found.")

CRITICAL ERROR: Code path /kaggle/input/chakramodel-kaggle-code not found.


## 2. Load Weights

In [2]:
# Link Weights - Updated to match the real diagnostic path
WEIGHTS_SRC = '/kaggle/input/datasets/gokulraj324/chakramodel-weightsupdated4'
!mkdir -p weights
if os.path.exists(WEIGHTS_SRC):
    !cp -r {WEIGHTS_SRC}/* weights/
    print("Weights loaded successfully.")
else:
    print(f"WARNING: Weights path {WEIGHTS_SRC} not found.")

Weights loaded successfully.


## 3. Video Inference & Temporal Proof (FPS Logging)

In [3]:
import glob

# Find any mp4 videos attached to the notebook
video_files = glob.glob("/kaggle/input/**/*.mp4", recursive=True)
if not video_files:
    print("WARNING: No mp4 videos found in /kaggle/input.")
else:
    print(f"Found {len(video_files)} video files. Running inference on the first 5 videos...\n")
    for vid in video_files[:5]:
        print(f"Evaluating: {vid}")
        !python src/infer_stream.py --source "{vid}" --model combo1 --weights weights/combo1_best.pth --yolo weights/best.pt

## 4. Full-Cohort Cross-Dataset Evaluation (OOD Truth)

In [4]:
import sys
import cv2
import numpy as np
import json
from tqdm.auto import tqdm

sys.path.append('/kaggle/working/chakramodel/src')
try:
    from chakranet_segmenter import ChakraNet
    from metrics.seg_metrics import dice as binary_dice_coefficient, iou as binary_iou
    segmenter = ChakraNet(device="cpu")
except Exception as e:
    print(f"Failed to load segmenter: {e}")
    segmenter = None

def eval_folder(img_dir, mask_dir):
    img_paths = [os.path.join(img_dir, f) for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    results = []
    for img_path in tqdm(img_paths, desc=os.path.basename(os.path.dirname(img_dir))):
        basename = os.path.basename(img_path)
        mask_path = os.path.join(mask_dir, basename)
        if not os.path.exists(mask_path):
            mask_path = os.path.join(mask_dir, os.path.splitext(basename)[0] + '.png')
            if not os.path.exists(mask_path):
                mask_path = os.path.join(mask_dir, os.path.splitext(basename)[0] + '.jpg')
        if not os.path.exists(mask_path): continue
        img = cv2.imread(img_path)
        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if img is None or gt_mask is None: continue
        gt_mask = (gt_mask > 127).astype(np.uint8)
        pred_mask, _, _, _ = segmenter.segment_roi(img, threshold=0.45, mc_passes=1)
        if pred_mask is None: continue
        pred_mask = cv2.resize(pred_mask, (gt_mask.shape[1], gt_mask.shape[0]), interpolation=cv2.INTER_NEAREST)
        pred_mask_bin = (pred_mask > 127).astype(np.uint8)
        results.append({"image": basename, "dice": binary_dice_coefficient(pred_mask_bin, gt_mask), "iou": binary_iou(pred_mask_bin, gt_mask)})
    if not results: return {"images": 0, "mean_dice": 0, "mean_iou": 0}
    return {"images": len(results), "mean_dice": float(np.mean([r["dice"] for r in results])), "mean_iou": float(np.mean([r["iou"] for r in results]))}

final_metrics = {}
if segmenter:
    print("\nScanning /kaggle/input for image/mask pairs...")
    for root, dirs, files in os.walk("/kaggle/input"):
        if "images" in dirs and "masks" in dirs:
            dataset_name = os.path.basename(root)
            # Avoid duplicate checks if nested
            if dataset_name not in final_metrics:
                print(f"\nEvaluating found dataset: {dataset_name}...")
                final_metrics[dataset_name] = eval_folder(os.path.join(root, "images"), os.path.join(root, "masks"))

    !mkdir -p results
    with open("results/final_proof_eval.json", "w") as f:
        json.dump(final_metrics, f, indent=4)
    print("\nFINAL PROOF METRICS:\n", json.dumps(final_metrics, indent=4))


Failed to load segmenter: No module named 'chakranet_segmenter'


## 5. Artifact Packaging

In [5]:
%cd /kaggle/working
!zip -r final_proof_artifacts.zip chakramodel/results chakramodel/outputs > /dev/null
print("\nDone! You can now download final_proof_artifacts.zip from the output data panel.")

/kaggle/working

Done! You can now download final_proof_artifacts.zip from the output data panel.
